# Phase 2 — SFT LoRA puis DPO de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA, puis alignement par préférences (DPO) du modèle SFT.

- **Prérequis** : GPU, fichiers `data/anonymized/sft_*_{train,validation,test}.anonymized.jsonl` produits par `data_prep_MediQAl` (français, réponse unique), `data_prep_SFT_MediQAl_mcqm` (français, réponses multiples) et `data_prep_SFT_UltraMedical` (anglais).
- **Prérequis DPO** : fichiers `data/anonymized/dpo_mediqal_mcqm_*` et `dpo_ultramedical_*` produits par `data_prep_DPO_MediQAl_mcqm` et `data_prep_DPO_UltraMedical`.
- **Étapes** : données → modèle → LoRA → entraînement SFT → validation intermédiaire → DPO → validation DPO → modèle final fusionné.
- **Suivi** : MLflow sur DagsHub (`MLFLOW_BACKEND = "dagshub"`), consultable de partout et quelle que soit la machine d'entraînement. En local (`"local"`) : `mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet.

In [1]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow dagshub pandas

## 1. Configuration

In [2]:
import functools
import gc
import json
import math
import re
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import DatasetDict, concatenate_datasets, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Données SFT de la phase 1, anonymisées : fichier -> langue
# Le SFT couvre les formats et les langues du DPO (section 7) : réponses uniques et listes en français, anglais
SFT_DATASETS = {
    "sft_mediqal_mcqu": "fr",   # QCM à réponse unique, avec cas clinique
    "sft_mediqal_mcqm": "fr",   # QCM à réponses multiples (liste), sans cas clinique
    "sft_ultramedical": "en",   # questions et réponses rédigées en anglais
}

# Suivi MLflow : "dagshub" (serveur distant) ou "local" (fichier ../mlflow.db)
MLFLOW_BACKEND = "dagshub"
DAGSHUB_REPO_OWNER = "obiyohan"
DAGSHUB_REPO_NAME = "my-ml-mlflow"

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = False  # True pour un run rapide sur 100 exemples, False pour le run complet

SEED = 42
RUN_NAME = f"sft-lora-r16-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda, dtype=torch.bfloat16


## 2. Données : train / validation / test

In [3]:
# Format prompt / completion : la loss ne porte que sur la réponse de l'assistant
def to_prompt_completion(record):
    return {"prompt": record["messages"][:-1], "completion": record["messages"][-1:]}


# Chaque source garde son découpage train / validation / test ; les sources sont ensuite mélangées
parts = {split: [] for split in ("train", "validation", "test")}
for stem, lang in SFT_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in parts}
    ds = load_dataset("json", data_files=files).select_columns(["messages"])
    for split, part in ds.items():
        parts[split].append(part.add_column("source", [stem] * len(part)).add_column("lang", [lang] * len(part)))

splits = DatasetDict({split: concatenate_datasets(p).shuffle(seed=SEED) for split, p in parts.items()})
splits = splits.map(to_prompt_completion, remove_columns=["messages"])

# Petit run : sous-échantillon (toutes sources mélangées)
if SMALL_RUN:
    for name, size in {"train": 50, "validation": 15, "test": 24}.items():
        splits[name] = splits[name].select(range(size))

pd.DataFrame({name: pd.Series(ds["source"]).value_counts() for name, ds in splits.items()}).fillna(0).astype(int)

Generating train split: 3935 examples [00:00, 118637.32 examples/s]
Generating validation split: 999 examples [00:00, 124306.09 examples/s]
Generating test split: 1996 examples [00:00, 124578.96 examples/s]
Map: 100%|██████████| 2996/2996 [00:00<00:00, 10555.24 examples/s]


,train,validation,test
sft_mediqal_mcqm,1000,200,500
sft_mediqal_mcqu,3935,999,1996
sft_ultramedical,1500,300,500


## 3. Tokenizer et modèle

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML, avec une réponse de l'assistant terminée par <|endoftext|> (eos_token) au lieu de <|im_end|>.
# Dans Qwen3-Base, <|im_start|> et <|im_end|> n'ont jamais été entraînés (embeddings à leur valeur d'initialisation)
# et LoRA ne modifie pas les embeddings : le modèle ne peut pas apprendre à produire <|im_end|>, donc à s'arrêter.
# <|endoftext|> a été appris au pré-entraînement : le modèle sait le prédire, et vLLM s'arrête dessus par défaut.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] }}"
    "{% if message['role'] == 'assistant' %}{{ eos_token }}{% else %}{{ '<|im_end|>\n' }}{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

example = splits["train"][0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

<|im_start|>system
Tu es un assistant médical expert. Réponds de façon précise et concise.<|im_end|>
<|im_start|>user
Vous êtes appelé d'urgence auprès d'un homme de 35 ans, manoeuvre dans une entreprise de travaux publics pour température à 40° C, point de côté droit, dyspnée. L'interrogatoire révèle un éthylisme chronique. Un tabagisme habituel, aucun problème pulmonaire jusqu'à ce jour. L'examen met en évidence des frissons et un syndrome de condensation pulmonaire apical droit

Quel diagnostic évoquez-vous ?<|im_end|>
<|im_start|>assistant
Pneumonie lobaire aiguë<|endoftext|>


In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:02<00:00, 126.47it/s]


1.72 B paramètres


## 4. LoRA

In [6]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [7]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    # Jeton lu dans la variable d'environnement DAGSHUB_USER_TOKEN,
    # sinon connexion via le navigateur au premier lancement (jeton mémorisé ensuite)
    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"Suivi MLflow : {mlflow.get_tracking_uri()}")

mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "small_run": SMALL_RUN, "device": device, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    num_train_epochs=2,
    max_steps=30 if SMALL_RUN else -1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # batch effectif = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1024,
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=10 if SMALL_RUN else 100,
    save_steps=10 if SMALL_RUN else 100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5 if SMALL_RUN else 20,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Accessing as ObiYohan

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Suivi MLflow : https://dagshub.com/obiyohan/my-ml-mlflow.mlflow


Truncating train dataset: 100%|██████████| 6435/6435 [00:00<00:00, 7092.07 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 6435/6435 [00:00<00:00, 32364.23 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 1499/1499 [00:00<00:00, 31984.69 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [8]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Training Loss,Validation Loss,Step,Before Loss,Entropy,Mean Token Accuracy,Num Tokens
No log,No log,0,1.382163,1.348127,0.667420,0.000000


{'val_before_loss': 1.382163166999817,
 'eval_entropy': 1.348127432326053,
 'eval_mean_token_accuracy': 0.6674199583048516,
 'eval_num_tokens': 0.0}

In [9]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,1.049686,1.119669,1.142760,0.708780,526949.000000
200,1.005782,1.099105,1.092428,0.713577,1047588.000000
300,0.928444,1.084693,1.093642,0.716058,1572027.000000
400,0.928277,1.075243,1.068300,0.717848,2085447.000000
500,0.872522,1.077653,0.974586,0.718555,2609919.000000
600,0.851506,1.072500,0.965142,0.719530,3129622.000000
700,0.839947,1.069625,0.971091,0.719956,3644587.000000
800,0.873867,1.069380,0.968634,0.719525,4168360.000000
806,0.873867,1.069628,0.968714,0.719623,4192954.000000


{'train_runtime': 3916.4082,
 'train_samples_per_second': 3.286,
 'train_steps_per_second': 0.206,
 'total_flos': 6.175705048055194e+16,
 'train_loss': 0.9045904570122807,
 'epoch': 2.0}

In [10]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data": {name: len(ds) for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

4574

## 6. Validation intermédiaire

1. **Loss, perplexité et token accuracy** sur validation (avant / après SFT) et test.
2. **Qualité des réponses générées** sur des exemples de test, modèle de base vs modèle SFT : accuracy (réponse exacte) et F1. Génération par lots pour bien charger le GPU.

La **latence** n'est pas mesurée ici : avec l'adaptateur LoRA non fusionné et Transformers, elle ne serait pas représentative. Elle sera mesurée en phase 3, sur le modèle final fusionné servi par vLLM.

In [11]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

# token_accuracy : part des tokens de la réponse attendue que le modèle prédit correctement
# (TRL la nomme « eval_mean_token_accuracy » quel que soit le préfixe : on la renomme ici)
evaluations = {"val_before": val_before, "val_after": val_after, "test": test_metrics}
summary = pd.DataFrame({
    name: {"loss": m[f"{name}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
    for name, m in evaluations.items()
}).T
summary["perplexity"] = summary["loss"].map(math.exp)

# Même mesure sur le test, source par source : chaque format doit être appris
per_source = {}
for source in SFT_DATASETS:
    subset = splits["test"].filter(lambda example: example["source"] == source)
    if len(subset):
        m = trainer.evaluate(subset, metric_key_prefix=f"test_{source}")
        per_source[f"test_{source}"] = {"loss": m[f"test_{source}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
per_source_summary = pd.DataFrame(per_source).T
per_source_summary["perplexity"] = per_source_summary["loss"].map(math.exp)
pd.concat([summary, per_source_summary])

Training Loss,Validation Loss,Step,After Loss,Entropy,Mean Token Accuracy,Num Tokens
0.873867,No log,806,1.069380,0.968634,0.719525,4192954.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 2996/2996 [00:00<00:00, 34446.45 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.873867,1.206900,806,1.061258,0.697442,4192954.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 1996/1996 [00:00<00:00, 32410.64 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Mcqu Loss,Entropy,Mean Token Accuracy,Num Tokens
0.873867,No log,806,2.011768,1.712359,0.582201,4192954.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 500/500 [00:00<00:00, 49798.21 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Mcqm Loss,Entropy,Mean Token Accuracy,Num Tokens
0.873867,No log,806,1.877314,1.652393,0.602052,4192954.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 500/500 [00:00<00:00, 21272.31 examples/s]


Training Loss,Validation Loss,Step,Sft Ultramedical Loss,Entropy,Mean Token Accuracy,Num Tokens
0.873867,No log,806,0.802895,0.723297,0.756297,4192954.000000


,loss,token_accuracy,perplexity
val_before,1.382163,0.667420,3.983509
val_after,1.069380,0.719525,2.913571
test,1.206900,0.697442,3.343104
test_sft_mediqal_mcqu,2.011768,0.582201,7.476527
test_sft_mediqal_mcqm,1.877314,0.602052,6.535926
test_sft_ultramedical,0.802895,0.756297,2.231994


In [12]:
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [IM_END_ID, tokenizer.eos_token_id]
# Réponses attendues sur le test : 57 tokens au plus (9 en moyenne).
# Le modèle de base, qui ne s'arrête pas toujours, est ainsi coupé plus tôt.
MAX_NEW_TOKENS = 64


def render(prompt):
    return tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate_batch(prompts, batch_size=16, max_new_tokens=MAX_NEW_TOKENS, desc=None, model=None):
    # Génération par lots (GPU bien chargé) : renvoie les réponses, dans l'ordre des prompts
    model = model or trainer.model  # par défaut, le modèle du SFT
    texts = [render(prompt) for prompt in prompts]
    # Prompts triés par longueur : moins de padding dans chaque lot, les plus longs d'abord
    # (un manque de mémoire GPU apparaît dès le premier lot)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    answers = [None] * len(texts)
    padding_side, tokenizer.padding_side = tokenizer.padding_side, "left"  # padding à gauche : la génération suit chaque prompt
    try:
        for start in tqdm(range(0, len(order), batch_size), desc=desc, unit="lot"):
            batch = order[start:start + batch_size]
            inputs = tokenizer([texts[i] for i in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id,
            )
            decoded = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, answer in zip(batch, decoded):
                answers[i] = answer.strip()
    finally:
        tokenizer.padding_side = padding_side
    return answers


def words(text):
    return re.findall(r"\w+", text.lower())


def exact_match(prediction, reference):
    # Accuracy : 1 si la réponse générée est identique à la réponse attendue (casse et ponctuation ignorées)
    return float(words(prediction) == words(reference))


def token_f1(prediction, reference):
    # Recouvrement de mots entre réponse générée et réponse attendue (0 à 1)
    pred, ref = words(prediction), words(reference)
    common = sum((Counter(pred) & Counter(ref)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(pred), common / len(ref)
    return 2 * precision * recall / (precision + recall)

In [13]:
N_EVAL = 20 if SMALL_RUN else 100    # exemples de test pour accuracy et F1
BATCH_SIZE = 16                      # à baisser en cas de manque de mémoire GPU (OutOfMemoryError)

# Sources évaluées par génération, avec leur nombre maximal de tokens générés.
# Les réponses anglaises (plusieurs centaines de tokens, rédigées) ne se comparent pas mot à mot à une référence :
# elles sont évaluées par la loss et la perplexité (cellule précédente).
GENERATION_SOURCES = {"sft_mediqal_mcqu": MAX_NEW_TOKENS, "sft_mediqal_mcqm": 192}

trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération

# 1. Qualité : génération par lots, modèle de base puis modèle SFT, source par source
source_generations = []
for source, max_new_tokens in GENERATION_SOURCES.items():
    subset = splits["test"].filter(lambda example: example["source"] == source)
    subset = subset.select(range(min(N_EVAL, len(subset))))
    prompts = list(subset["prompt"])
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"Base {source}")
    sft_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"SFT {source}")
    source_generations.append(pd.DataFrame({
        "source": source,
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": [completion[0]["content"] for completion in subset["completion"]],
        "base": base_answers,
        "sft": sft_answers,
    }))
generations = pd.concat(source_generations, ignore_index=True)
references = generations["reference"]
for model_name in ("base", "sft"):
    generations[f"accuracy_{model_name}"] = [exact_match(a, r) for a, r in zip(generations[model_name], references)]
    generations[f"f1_{model_name}"] = [token_f1(a, r) for a, r in zip(generations[model_name], references)]
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)

# Moyennes sur les exemples de test : modèle de base vs modèle SFT
QUALITY_METRICS = ["accuracy", "f1"]
comparison = pd.DataFrame({
    model_name: {metric: generations[f"{metric}_{model_name}"].mean() for metric in QUALITY_METRICS}
    for model_name in ("base", "sft")
})

# Qualité par source
quality_by_source = generations.groupby("source")[
    [f"{metric}_{model_name}" for metric in QUALITY_METRICS for model_name in ("base", "sft")]
].mean()
display(quality_by_source)
comparison

SFT sft_mediqal_mcqm: 100%|██████████| 7/7 [01:32<00:00, 13.26s/lot]


,accuracy_base,accuracy_sft,f1_base,f1_sft
source,,,,
sft_mediqal_mcqm,0.0,0.0,0.042436,0.092654
sft_mediqal_mcqu,0.0,0.0,0.045573,0.122691


,base,sft
accuracy,0.000000,0.000000
f1,0.044005,0.107673


In [14]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations.groupby("source")[["source", "question", "reference", "base", "sft"]].head(3)

,source,question,reference,base,sft
0,sft_mediqal_mcqu,"olestérol, triglycérides, uricémie\n\nVous avez la possibilité d'obtenir un examen Doppler le lendemain. Parmi les renseignements suivants, lequel sera le plus important pour la décision thérapeut...",Etat des carotides et vertébrales,"Le plus important pour la décision thérapeutique est l'examen Doppler, car il permet de détecter une insuffisance circulatoire aiguë, qui pourrait être la cause de la douleur et de la plaquiste.",La vitesse de déplacement du flux sanguin dans le gros orteil gauche
1,sft_mediqal_mcqu,"lique de rétrécissement mitral.Un bilan est réalisé après traitement de l'accident aigu.\n\nL'échocardiogramme mettra en évidence les signes contenus dans les propositions suivantes sauf une, laqu...",Un fluttering de la grande valve mitrale,"La réponse est : ""La réponse est : ""La réponse est : ""La réponse est : ""La réponse est : ""La réponse est : ""La réponse est : ""La réponse est : ""La réponse est : ""La réponse est : ""La réponse est :...",Une augmentation de la vitesse de débit mitral
2,sft_mediqal_mcqu,"ans complication et dans les antécédents familiaux, une tante maternelle asthmatique\n\nQuel(s) examen(s) est-il impératif de pratiquer en urgence avant toute décision thérapeutique parmi les suiv...",Aucun,<System> : Tu es un assistant médical expert. Réponds de façon précise et concise..AutoScaleMode\n<System> : Tu es un assistant médical expert. Réponds de façon précise et concise..AutoScaleMode\n...,Radiographie pulmonaire
100,sft_mediqal_mcqm,"Un médecin praticien qui dans l'exercice de sa profession constate un cas de sévices contre mineur de moins de 15 ans peut, sans déroger au secret professionnel, le dénoncer en informant :",- Le médecin inspecteur départemental de la D A S S\n- Le directeur départemental de la D A S S.\n- L'assistance sociale polyvalente de secteur,"Un médecin praticien qui dans l'exercice de sa profession constate un cas de sévices contre mineur de moins de 15 ans peut, sans déroger au secret professionnel, le dénoncer en informant :��态\n��态...",- La police municipale\n- Le préfet de région\n- Le procureur de la République
101,sft_mediqal_mcqm,"Parmi les éléments suivants, lequel(lesquels) caractérise(nt) le carcinome spino-cellulaire différencié ?",- Des cordons dont les cellules sont unies par des ponts d'union\n- La présence de globes cornés,"Parmi les éléments suivants, lequel(lesquels) caractérise(nt) le carcinome spino-cellulaire différencié ?��态\n��态\nLe carcinome spino-cellulaire différencié est caractérisé par la présence de cel...",- Carcinome épidermoïde\n- Carcinome épidermoïde à cellules à corps de ver\n- Carcinome épidermoïde à cellules à corps de ver à corps de ver
102,sft_mediqal_mcqm,"Parmi les médicaments suivants, lesquels ne doivent pas être associés aux anticoagulants antivitamine K ?",- Miconazole (Daktarin®)\n- Clofibrate (Lipavlon®)\n- Phénylbutazone,"Parmi les médicaments suivants, lesquels ne doivent pas être associés aux anticoagulants antivitamine K ?.RemoveEmptyEntries\n.RemoveEmptyEntries\n.RemoveEmptyEntries\n.RemoveEmptyEntries\n.Remove...",- Aspirine\n- Acide acétylsalicylique\n- Ibuprofène\n- Diclofenac


In [15]:
# Métriques finales dans MLflow (les loss sont déjà envoyées par le Trainer), puis clôture du run
mlflow.log_metrics({f"{name}_token_accuracy": row["token_accuracy"] for name, row in summary.iterrows()})
mlflow.log_metrics({f"{name}_perplexity": row["perplexity"] for name, row in summary.iterrows()})
mlflow.log_metrics({
    f"test_{metric}_{model}": comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in comparison.columns
})
# Par source : loss, token accuracy et perplexité du test, accuracy et F1 des réponses générées
mlflow.log_metrics({
    f"{name}_{metric}": value for name, row in per_source_summary.iterrows() for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_{source}_{column}": value for source, row in quality_by_source.iterrows() for column, value in row.items()
})
mlflow.end_run()

🏃 View run sft-lora-r16-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2/runs/15457dfc3af044b29113043902166cfd
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2


## Lecture des résultats

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | courbes de loss d'entraînement et de validation | baissent ensemble ; si `eval_loss` remonte : surapprentissage |
| `val_before_loss`, `val_after_loss`, `test_loss` | loss avant / après SFT et sur le test | après < avant ; test proche de val |
| `*_token_accuracy` | part des tokens de la réponse correctement prédits | augmente après SFT |
| `*_perplexity` | exp(loss) | baisse après SFT |
| `test_sft_<source>_loss`, `_token_accuracy`, `_perplexity` | les mêmes mesures sur le test, source par source | chaque source s'améliore, y compris l'anglais |
| `test_accuracy_*` | réponse générée identique à la réponse attendue (sources françaises) | sft > base |
| `test_f1_*` | recouvrement de mots avec la réponse attendue | sft > base |
| `test_sft_<source>_{accuracy,f1}_*` | accuracy et F1, source par source | sft > base pour `mcqu` et `mcqm` |

- La courbe `eval_mean_token_accuracy` mélange toutes les évaluations (TRL ignore le préfixe) : utiliser `*_token_accuracy`.
- Accuracy et F1 restent sévères : réponses libres, sans les propositions du QCM.
- La latence (TTFT, TPOT, débit) sera mesurée en phase 3, sur le modèle final servi par vLLM.

**Suite** : passer `SMALL_RUN = False` pour l'entraînement complet, puis DPO (section 7).

## 7. Alignement DPO

Le DPO part du **modèle SFT** et lui apprend à préférer la réponse `chosen` à la réponse `rejected` de chaque paire.

- **Modèle de départ** : modèle de base + adaptateur SFT **fusionné** (les poids LoRA sont ajoutés aux poids du modèle). TRL calcule les log-probabilités directement depuis `lm_head`, ce qui impose de fusionner l'adaptateur SFT avant d'en ajouter un nouveau.
- **Nouvel adaptateur LoRA** entraîné par DPO. La **référence** du DPO est le modèle SFT lui-même : TRL l'obtient en désactivant l'adaptateur, sans charger un second modèle en mémoire.
- **Données** : paires françaises (MediQAl `mcqm`, vérifiées par le corrigé) et anglaises (UltraMedical, filtrées sur la qualité), mélangées pour l'entraînement, évaluées séparément par langue.
- **Reprise après redémarrage** : exécuter d'abord la section 1 (configuration), puis cette section. Le tokenizer et l'adaptateur SFT sont relus depuis `SFT_ADAPTER_DIR`. Avant la validation (7.4), exécuter aussi la cellule des fonctions de génération (section 6).

In [16]:
# Paires DPO de la phase 1, par langue
DPO_DATASETS = {"fr": "dpo_mediqal_mcqm", "en": "dpo_ultramedical"}
DPO_COLUMNS = ["source", "lang", "pair_check", "prompt", "chosen", "rejected"]  # colonnes communes aux deux jeux

# Hyperparamètres DPO
DPO_BETA = 0.1              # écart toléré avec la référence : plus beta est grand, plus le modèle reste proche du SFT
DPO_LEARNING_RATE = 5e-6    # valeur de départ usuelle en DPO ; à augmenter si rewards/accuracies stagne vers 0,5
DPO_MAX_LENGTH = 1024       # prompt + réponse, en tokens
DPO_N_EVAL = 20 if SMALL_RUN else 100  # exemples du test français pour la génération (7.4)
DPO_BATCH_SIZE = 16                    # génération par lots : à baisser en cas d'OutOfMemoryError

SFT_ADAPTER_DIR = Path("../outputs") / RUN_NAME / "final"  # adaptateur SFT de la section 5
DPO_RUN_NAME = f"dpo-lora-r16-beta{DPO_BETA}-lr{DPO_LEARNING_RATE}-seed{SEED}" + ("-small" if SMALL_RUN else "")
DPO_OUTPUT_DIR = Path("../outputs") / DPO_RUN_NAME  # checkpoints, adaptateur DPO et modèle final fusionné
print(f"SFT : {SFT_ADAPTER_DIR}\nDPO : {DPO_OUTPUT_DIR}")

SFT : ../outputs/sft-lora-r16-seed42/final
DPO : ../outputs/dpo-lora-r16-beta0.1-lr5e-06-seed42


### 7.1 Données DPO

In [17]:
# Tokenizer sauvegardé avec l'adaptateur SFT : même gabarit qu'à l'entraînement SFT
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR)


def pair_length(example):
    # Longueur en tokens de la plus longue des deux conversations (prompt + chosen, prompt + rejected)
    return max(
        len(tokenizer(tokenizer.apply_chat_template(example["prompt"] + answer, tokenize=False), add_special_tokens=False)["input_ids"])
        for answer in (example["chosen"], example["rejected"])
    )


dpo_by_lang = {}
for lang, stem in DPO_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in ("train", "validation", "test")}
    ds = load_dataset("json", data_files=files).select_columns(DPO_COLUMNS)
    # Paires trop longues : TRL couperait la fin de la réponse, on les écarte
    ds = ds.filter(lambda example: pair_length(example) <= DPO_MAX_LENGTH)
    if SMALL_RUN:
        for split, size in {"train": 32, "validation": 8, "test": 8}.items():
            ds[split] = ds[split].shuffle(seed=SEED).select(range(size))
    dpo_by_lang[lang] = ds
    print(f"{lang}: " + ", ".join(f"{split} {len(d)}" for split, d in ds.items()))

# Entraînement et validation : les deux langues mélangées
dpo_train = concatenate_datasets([ds["train"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
dpo_validation = concatenate_datasets([ds["validation"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
print(f"DPO : train {len(dpo_train)}, validation {len(dpo_validation)}")

Filter: 100%|██████████| 1320/1320 [00:01<00:00, 920.62 examples/s]


fr: train 2000, validation 300, test 1320


Filter: 100%|██████████| 756/756 [00:02<00:00, 332.80 examples/s]

en: train 1988, validation 191, test 750
DPO : train 3988, validation 491


In [18]:
# Une paire telle que la verra le modèle
example = dpo_train[0]
print(tokenizer.apply_chat_template(example["prompt"], add_generation_prompt=True, tokenize=False))
print(f"--- CHOSEN :\n{example['chosen'][0]['content']}\n--- REJECTED :\n{example['rejected'][0]['content']}")

<|im_start|>system
You are an expert medical assistant. Answer accurately and clearly.<|im_end|>
<|im_start|>user
In the context of esophageal cancer treatment, assuming EUS indicates aortic involvement without distant metastases and the patient's performance status allows for aggressive therapy, what is the most appropriate subsequent action per current guidelines?

A. Surgical resection may be considered absent malignant pleural effusion.
B. Initiate induction chemoradiation therapy.
C. Surgical resection is generally contraindicated.
D. Proceed with esophagectomy including regional lymphadenectomy.<|im_end|>
<|im_start|>assistant

--- CHOSEN :
In the context of esophageal cancer treatment, the patient has aortic involvement without distant metastases and a good performance status, allowing for aggressive therapy. 

Aortic involvement is a significant factor, as it indicates T4 disease, which is a locally advanced tumor. In this scenario, surgical resection may not be the most effect

### 7.2 Modèle de départ : modèle SFT fusionné

In [19]:
# Libère la mémoire GPU du SFT si la section 5 vient de tourner
if "trainer" in globals():
    del trainer, model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Modèle de base + adaptateur SFT fusionné : point de départ et référence du DPO
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
print(f"{sft_model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1623.91it/s]


1.72 B paramètres


### 7.3 Entraînement DPO

- **LoRA** : même configuration que pour le SFT, sur un nouvel adaptateur.
- **Une époque** : le DPO sur-apprend vite. Au-delà, le modèle mémorise les paires sans mieux généraliser.
- **Batch effectif de 16** (2 × 8) : chaque exemple contient deux réponses, donc deux fois plus de tokens qu'en SFT.
- **Learning rate faible** et montée progressive sur 10 % des étapes : le DPO déstabilise facilement le modèle.

In [20]:
dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# Suivi MLflow : même serveur que le SFT (connexion faite en section 5), expérience dédiée au DPO
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("qwen3-1.7b-dpo-lora")
mlflow.start_run(run_name=DPO_RUN_NAME)
mlflow.log_params({
    "model": MODEL_NAME, "sft_adapter": str(SFT_ADAPTER_DIR), "small_run": SMALL_RUN, "device": device,
    "beta": DPO_BETA, "lora_r": dpo_lora_config.r, "lora_alpha": dpo_lora_config.lora_alpha,
    **{f"pairs_{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
})

dpo_args = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    run_name=DPO_RUN_NAME,
    seed=SEED,
    # DPO
    beta=DPO_BETA,
    loss_type=["sigmoid"],  # perte DPO standard
    max_length=DPO_MAX_LENGTH,
    # Optimisation
    num_train_epochs=1,
    max_steps=10 if SMALL_RUN else -1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # batch effectif = 16 paires
    learning_rate=DPO_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,  # 10 % des étapes en montée progressive du learning rate
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=5 if SMALL_RUN else 25,
    save_steps=5 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=1 if SMALL_RUN else 5,
    report_to="mlflow",
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_validation,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)
dpo_trainer.model.print_trainable_parameters()

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

2026/11/10 17:50:42 INFO mlflow.tracking.fluent: Experiment with name 'qwen3-1.7b-dpo-lora' does not exist. Creating a new experiment.
Tokenizing train dataset: 100%|██████████| 3988/3988 [00:11<00:00, 358.74 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 3988/3988 [00:00<00:00, 38793.01 examples/s]
Tokenizing eval dataset: 100%|██████████| 491/491 [00:01<00:00, 358.20 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 491/491 [00:00<00:00, 30896.46 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [21]:
# Reprise après interruption : dpo_trainer.train(resume_from_checkpoint=True)
dpo_result = dpo_trainer.train()
dpo_result.metrics

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
25,0.700269,0.696177,0.910159,308695.000000,3.805323,3.706482,0.722251,-0.003717,0.000007,0.497967,-0.003724,-125.080621,-129.695865
50,0.691809,0.690750,0.910765,633402.000000,3.813311,3.715041,0.722140,0.001971,-0.005132,0.514228,0.007104,-125.023736,-129.747254
75,0.677630,0.679885,0.911916,954140.000000,3.810414,3.712080,0.723044,0.008086,-0.021840,0.584011,0.029927,-124.962589,-129.914335
100,0.663943,0.669614,0.912459,1258839.000000,3.797660,3.700340,0.723045,0.011808,-0.041240,0.617209,0.053048,-124.925365,-130.108326
125,0.634104,0.658421,0.913786,1584765.000000,3.794447,3.699999,0.723065,0.013645,-0.065652,0.657859,0.079297,-124.906999,-130.352455
150,0.664992,0.648750,0.914635,1906438.000000,3.782861,3.688936,0.722521,0.014202,-0.089579,0.681572,0.103781,-124.901434,-130.591726
175,0.624735,0.643778,0.915742,2225462.000000,3.775659,3.680521,0.723348,0.009147,-0.109590,0.676152,0.118737,-124.951980,-130.791829
200,0.630313,0.636507,0.916132,2536695.000000,3.772683,3.678377,0.722854,0.012368,-0.126428,0.690379,0.138796,-124.919767,-130.960211
225,0.600068,0.637499,0.916145,2862382.000000,3.768210,3.674572,0.722841,0.004937,-0.131444,0.698509,0.136380,-124.994084,-131.010367
250,0.656714,0.634533,0.916119,3160191.000000,3.771405,3.678426,0.722726,0.011399,-0.132123,0.692412,0.143522,-124.929461,-131.017160


{'train_runtime': 3204.7528,
 'train_samples_per_second': 1.244,
 'train_steps_per_second': 0.078,
 'total_flos': 3.579274673897472e+16,
 'train_loss': 0.6536764822006226,
 'epoch': 1.0}

In [22]:
# Sauvegarde de l'adaptateur DPO (meilleur checkpoint) et de la configuration du run
DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_trainer.save_model(str(DPO_FINAL_DIR))
tokenizer.save_pretrained(str(DPO_FINAL_DIR))

dpo_run_config = {
    "model": MODEL_NAME,
    "sft_adapter": str(SFT_ADAPTER_DIR),  # l'adaptateur DPO s'applique au modèle SFT fusionné
    "seed": SEED,
    "small_run": SMALL_RUN,
    "beta": DPO_BETA,
    "lora": {"r": dpo_lora_config.r, "alpha": dpo_lora_config.lora_alpha, "dropout": dpo_lora_config.lora_dropout},
    "data": {f"{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
    "training_args": dpo_args.to_dict(),
}
(DPO_FINAL_DIR / "run_config.json").write_text(json.dumps(dpo_run_config, indent=2, default=str), encoding="utf-8")

4889

### 7.4 Validation DPO

1. **Préférences** sur la validation et le test de chaque langue. La *récompense implicite* d'une réponse mesure combien le modèle DPO la juge plus probable que le modèle SFT. On en tire :
   - `reward_accuracy` : part des paires où la récompense de la `chosen` dépasse celle de la `rejected` (0,5 = hasard) ;
   - `reward_margin` : écart moyen entre les deux récompenses ;
   - `logps_chosen` / `logps_rejected` : log-probabilités moyennes des deux réponses sous le modèle DPO.
2. **Réponses générées** sur le test français : modèle SFT vs modèle DPO, avec l'accuracy (réponse exacte) et le F1 de la section 6. Le test `mcqm` a un corrigé : on compare chaque réponse à la liste des bonnes réponses.

In [23]:
# 1. Préférences, par split et par langue
# (TRL nomme ses métriques « eval_rewards/... » quel que soit le préfixe : on les renomme ici)
dpo_rows = {}
for lang, ds in dpo_by_lang.items():
    for split in ("validation", "test"):
        name = f"{split}_{lang}"
        m = dpo_trainer.evaluate(ds[split], metric_key_prefix=name)
        dpo_rows[name] = {
            "loss": m[f"{name}_loss"],
            "reward_accuracy": m["eval_rewards/accuracies"],
            "reward_margin": m["eval_rewards/margins"],
            "logps_chosen": m["eval_logps/chosen"],
            "logps_rejected": m["eval_logps/rejected"],
        }
dpo_summary = pd.DataFrame(dpo_rows).T
dpo_summary

Tokenizing eval dataset: 100%|██████████| 300/300 [00:00<00:00, 488.96 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 300/300 [00:00<00:00, 28956.86 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.656714,No log,250,0.688580,1.683069,3160191.000000,4.710877,4.725655,0.597474,0.010024,0.000452,0.580000,0.009572,-64.211679,-65.954541


Tokenizing eval dataset: 100%|██████████| 1320/1320 [00:02<00:00, 492.85 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 1320/1320 [00:00<00:00, 47747.63 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.656714,No log,250,0.688321,1.745439,3160191.000000,4.726042,4.720754,0.577368,0.008016,-0.002107,0.600000,0.010123,-63.948573,-63.573956


Tokenizing eval dataset: 100%|██████████| 191/191 [00:00<00:00, 251.26 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 191/191 [00:00<00:00, 15166.83 examples/s]


Training Loss,Validation Loss,Step,En Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.656714,No log,250,0.554605,0.613373,3160191.000000,3.309077,3.221983,0.780795,0.006812,-0.338070,0.822917,0.344882,-219.544515,-233.046635


Tokenizing eval dataset: 100%|██████████| 750/750 [00:03<00:00, 244.75 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 750/750 [00:00<00:00, 23518.05 examples/s]


Training Loss,Validation Loss,Step,En Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.656714,No log,250,0.682456,0.694709,3160191.000000,3.435534,3.216857,0.756287,-0.092327,-0.147491,0.514628,0.055165,-344.768579,-248.359152


,loss,reward_accuracy,reward_margin,logps_chosen,logps_rejected
validation_fr,0.688580,0.580000,0.009572,-64.211679,-65.954541
test_fr,0.688321,0.600000,0.010123,-63.948573,-63.573956
validation_en,0.554605,0.822917,0.344882,-219.544515,-233.046635
test_en,0.682456,0.514628,0.055165,-344.768579,-248.359152


In [24]:
# 2. Réponses générées sur le test français : modèle SFT (adaptateur DPO désactivé) vs modèle DPO
assert "generate_batch" in globals(), "Exécuter d'abord la cellule des fonctions de génération (section 6)"
DPO_MAX_NEW_TOKENS = 192  # réponses mcqm : listes de 2 à 5 options

dpo_model = dpo_trainer.model.eval()
dpo_model.config.use_cache = True  # accélère la génération

test_fr = dpo_by_lang["fr"]["test"].select(range(min(DPO_N_EVAL, len(dpo_by_lang["fr"]["test"]))))
fr_prompts = list(test_fr["prompt"])
fr_references = [chosen[0]["content"] for chosen in test_fr["chosen"]]

with dpo_model.disable_adapter():
    fr_sft_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération SFT", model=dpo_model)
fr_dpo_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération DPO", model=dpo_model)

dpo_generations = pd.DataFrame({
    "question": [prompt[-1]["content"][-200:] for prompt in fr_prompts],
    "reference": fr_references,
    "sft": fr_sft_answers,
    "dpo": fr_dpo_answers,
})
for model_name in ("sft", "dpo"):
    dpo_generations[f"accuracy_{model_name}"] = [exact_match(a, r) for a, r in zip(dpo_generations[model_name], fr_references)]
    dpo_generations[f"f1_{model_name}"] = [token_f1(a, r) for a, r in zip(dpo_generations[model_name], fr_references)]
dpo_generations.to_csv(DPO_FINAL_DIR / "test_fr_generations.csv", index=False)

DPO_QUALITY_METRICS = ["accuracy", "f1"]
dpo_comparison = pd.DataFrame({
    model_name: {metric: dpo_generations[f"{metric}_{model_name}"].mean() for metric in DPO_QUALITY_METRICS}
    for model_name in ("sft", "dpo")
})
dpo_comparison

Génération DPO: 100%|██████████| 7/7 [00:19<00:00,  2.82s/lot]


,sft,dpo
accuracy,0.000000,0.000000
f1,0.065594,0.060103


In [25]:
# Quelques exemples à relire
dpo_generations[["question", "reference", "sft", "dpo"]].head(5)

,question,reference,sft,dpo
0,"n du carmin dans les selles est de 6 heures. La NFS indique 130 g d'Hb/l et un volume érythrocytaire de 110 µ3\n\nParmi les maladies suivantes, quelle(s) est(sont) la(les) cause(s) de diarrhée mot...",- Hyperthyroïdie\n- Tumeur carcinoïde entraînant un syndrome carcinoïde\n- Cancer médullaire du corps thyroïde\n- Déconjugaision des sels biliaires dans l'iléon\n- Diabète avec neuropathie végétative,Malabsorption intestinale,Malabsorption intestinale
1,"émie : 660 micromol/l - Calcémie : 2,3 micromol/l Le myélogramme montre une moelle riche avec une plasmocytose à 34 %\n\nParmi les critères suivants, lequel ou lesquels a(ont) une valeur pronost...","- Hémoglobine : 81 g/l\n- Créatinémie : 180 micromol/l\n- Protéinurie de Bence-Jones : 4,3 g/24 h.",Aucun de ces critères,Le taux de calcémie
2,"hospitalisation, traitée et surveillée\n\nAu bout de 24 heures, son état s'aggrave. Parmi les critères suivants, le(s)quel(s) vous paraît(paraissent) être des critères de gravité d'une crise d'as...",- L'absence de sédation sous ventoline correctement administrée\n- L'existence d'un silence auscultatoire\n- Une hypercapnie\n- La notion d'état de mal dans les antécédents,Toutes ces réponses sont justes,Toutes ces réponses sont justes
3,tion périphérique et périostose ; les articulations coxofémorales ont un aspect radiologique normal\n\nLe(s)quel(s) de ces 5 éléments peu(ven)t être incriminé(s) dans la survenue de cette ostéomal...,- Le régime alimentaire très pauvre en lipides d'origine animale\n- Un défaut d'exposition solaire,Hyperphosphatasémie alcaline,Hyperphosphatasémie alcaline
4,"il existe une diarrhée, et une élévation de la créatinine sérique (2 fois la limite supérieure de la normale)\n\nSur quel(s) élément(s) allez-vous juger de l'efficacité thérapeutique à la 48ème h...",- Evolution de la température\n- Gaz du sang artériel\n- Symptomatologie clinique,Aucun des éléments précédents,Aucun des éléments précédents


In [26]:
# Métriques finales dans MLflow, puis clôture du run
mlflow.log_metrics({
    f"{name}_{metric}": value
    for name, row in dpo_summary.iterrows()
    for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_fr_{metric}_{model}": dpo_comparison.loc[metric, model]
    for metric in DPO_QUALITY_METRICS
    for model in dpo_comparison.columns
})
mlflow.end_run()

🏃 View run dpo-lora-r16-beta0.1-lr5e-06-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3/runs/0627c48c97b24f46b7ffccacd8aaf59a
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3


### 7.5 Modèle final fusionné

Modèle de base + SFT + DPO en un seul jeu de poids complets (environ 3,4 Go en bf16), avec le tokenizer et son gabarit : c'est le modèle à servir avec vLLM en phase 3.

In [27]:
MERGED_DIR = DPO_OUTPUT_DIR / "merged"
merged_model = dpo_trainer.model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Modèle final : {MERGED_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.39s/it]

Modèle final : ../outputs/dpo-lora-r16-beta0.1-lr5e-06-seed42/merged


## Lecture des résultats DPO

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | perte DPO d'entraînement et de validation (0,69 = aucune préférence apprise) | baissent sous 0,69 ; si `eval_loss` remonte : surapprentissage |
| `rewards/accuracies` | part des paires où la `chosen` l'emporte sur la `rejected` | monte nettement au-dessus de 0,5 |
| `rewards/margins` | écart moyen de récompense entre `chosen` et `rejected` | augmente, sans exploser |
| `logps/chosen`, `logps/rejected` | log-probabilités moyennes des deux réponses | `rejected` baisse plus que `chosen` ; si `chosen` chute fortement, baisser le learning rate ou augmenter beta |
| `{validation,test}_{fr,en}_*` | les mêmes métriques, par split et par langue | proches entre validation et test, et entre langues |
| `test_fr_accuracy_*`, `test_fr_f1_*` | réponses générées sur le test `mcqm`, comparées au corrigé | dpo ≥ sft |

- Les courbes `eval_rewards/*` mélangent toutes les évaluations (TRL ignore le préfixe) : pour la comparaison finale, utiliser `{validation,test}_{fr,en}_*`.
- **Itérer** : un run par jeu d'hyperparamètres (`DPO_BETA`, `DPO_LEARNING_RATE`), avec le même `SEED`. Le nom du run les contient.

**Suite** : contrôles de sécurité (hallucinations, recommandations dangereuses) sur le modèle final, puis déploiement avec vLLM (phase 3).